# Stage 11 — DeepScoresV2 Dense Residual U-Net
Non-production training. Verifies official MD5 before extraction, preserves official test as held-out, prevents source-family leakage, synthesizes degraded inputs on the fly, checkpoints to Drive, and never downloads pretrained weights.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib,json,tarfile,time,random,math,io
A=Path("/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_TRAINING_DATA/ds2_dense.tar.gz")
O=Path("/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_TRAINING_OUTPUT/deepscoresv2_dense_residual_unet_v1"); O.mkdir(parents=True,exist_ok=True)
X=Path("/content/st_score_restore_deepscoresv2_dense")
MD5="7237318e381e6e0848ec30eb82decb83"; SIZE=741814529
if not A.exists():
    m=list(Path("/content/drive/MyDrive").rglob("ds2_dense.tar.gz"))
    if len(m)!=1: raise FileNotFoundError(f"Expected one ds2_dense.tar.gz, found {len(m)}")
    A=m[0]
if A.stat().st_size!=SIZE: raise RuntimeError("Archive size mismatch")
h=hashlib.md5()
with A.open("rb") as f:
    while b:=f.read(8<<20): h.update(b)
if h.hexdigest()!=MD5: raise RuntimeError(f"MD5 mismatch: {h.hexdigest()} != {MD5}")
(O/"archive_identity.json").write_text(json.dumps({"datasetId":"deepscoresv2.dense.v2","size":SIZE,"md5":MD5,"verified":True,"doi":"10.5281/zenodo.4012193","license":"CC-BY-4.0"},indent=2))

def safe_extract():
    X.mkdir(parents=True,exist_ok=True); base=X.resolve()
    with tarfile.open(A,"r:gz") as t:
        for m in t.getmembers():
            p=m.name.replace("\\\\","/")
            if not p or p.startswith("/") or ".." in Path(p).parts or m.issym() or m.islnk() or m.isdev(): raise RuntimeError(f"unsafe tar member: {m.name}")
            q=(X/p).resolve()
            if q!=base and base not in q.parents: raise RuntimeError(f"tar escape: {m.name}")
        t.extractall(X)
if not X.exists() or not any(X.iterdir()): safe_extract()
tr=list(X.rglob("deepscores_train.json")); te=list(X.rglob("deepscores_test.json"))
if len(tr)!=1 or len(te)!=1: raise RuntimeError("Expected one DeepScores train/test JSON")
R=tr[0].parent
def names(p):
    z=json.loads(p.read_text())
    if not isinstance(z,dict) or not isinstance(z.get("images"),list): raise RuntimeError("Unsupported annotation JSON")
    out=[]
    for r in z["images"]:
        n=r.get("file_name") or r.get("filename") or r.get("img_name")
        if n: out.append(str(n))
    return out
T,H=names(tr[0]),names(te[0])
if (len(T),len(H))!=(1362,352): raise RuntimeError(f"Unexpected counts {(len(T),len(H))}")
def fam(n): return Path(n).name.split("-aug-",1)[0] if "-aug-" in Path(n).name else Path(n).stem
HF={fam(n) for n in H}; T0=[n for n in T if fam(n) not in HF]
def dev(f): return int.from_bytes(hashlib.sha256(("st-score-restore-stage11-deepscoresv2-dense-v1:"+f).encode()).digest()[:4],"big")%100<10
TR=[n for n in T0 if not dev(fam(n))]; DV=[n for n in T0 if dev(fam(n))]
if not TR or not DV: raise RuntimeError("Empty derived split")
if not {fam(n) for n in TR}.isdisjoint({fam(n) for n in DV}|HF): raise RuntimeError("Source-family leakage")
(O/"dataset_manifest.json").write_text(json.dumps({"official":{"train":len(T),"heldOut":len(H)},"derived":{"train":len(TR),"development":len(DV),"heldOut":len(H),"leakageExcluded":len(T)-len(T0)},"heldOutTuningForbidden":True,"sourceFamilyLeakagePrevented":True},indent=2))
def path(n):
    cand=[R/"images"/n,R/n]
    for p in cand:
        if p.exists(): return p
    m=list(R.rglob(Path(n).name))
    if len(m)==1:return m[0]
    raise FileNotFoundError(n)
print("MD5 verified; splits:",len(TR),len(DV),len(H))

In [ ]:
import torch,torch.nn as nn,torch.nn.functional as F,numpy as np
from torch.utils.data import Dataset,DataLoader
from PIL import Image,ImageFilter
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode
if not torch.cuda.is_available(): raise RuntimeError("GPU required: Colab Runtime > Change runtime type > GPU")
D=torch.device("cuda"); SEED=20260907; PATCH=512; BATCH=4; EPOCHS=20; LR=2e-4
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
def rng(n,v): return random.Random(int.from_bytes(hashlib.sha256(f"{SEED}:{n}:{v}".encode()).digest()[:8],"big"))
def crop(im,r):
    w,h=im.size
    if min(w,h)<PATCH:
        s=max(PATCH/w,PATCH/h); im=im.resize((math.ceil(w*s),math.ceil(h*s)),Image.Resampling.BICUBIC); w,h=im.size
    a=np.asarray(im); best=(0,0); ink=-1
    for _ in range(8):
        x=r.randint(0,max(0,w-PATCH)); y=r.randint(0,max(0,h-PATCH)); q=(a[y:y+PATCH,x:x+PATCH]<235).mean()
        if q>ink: ink=q; best=(x,y)
    x,y=best; return im.crop((x,y,x+PATCH,y+PATCH))
def degrade(im,r):
    im=TF.rotate(im,r.uniform(-2.5,2.5),interpolation=InterpolationMode.BILINEAR,fill=255); w,h=im.size
    d=r.uniform(.002,.02); dx,dy=int(w*d),int(h*d)
    s=[[0,0],[w-1,0],[w-1,h-1],[0,h-1]]
    e=[[r.randint(0,dx),r.randint(0,dy)],[w-1-r.randint(0,dx),r.randint(0,dy)],[w-1-r.randint(0,dx),h-1-r.randint(0,dy)],[r.randint(0,dx),h-1-r.randint(0,dy)]]
    im=TF.perspective(im,s,e,interpolation=InterpolationMode.BILINEAR,fill=255)
    im=TF.adjust_brightness(im,r.uniform(.78,1.2)); im=TF.adjust_contrast(im,r.uniform(.82,1.18)); im=im.filter(ImageFilter.GaussianBlur(r.uniform(.15,1.75)))
    a=np.asarray(im).astype(np.float32)/255.; yy,xx=np.mgrid[:a.shape[0],:a.shape[1]]; cx,cy=r.uniform(0,a.shape[1]),r.uniform(0,a.shape[0]); z=np.sqrt((xx-cx)**2+(yy-cy)**2); z/=max(z.max(),1)
    a*=1-r.uniform(0,.22)*(1-z); a+=np.random.default_rng(r.randrange(2**32)).normal(0,r.uniform(.002,.03),a.shape); a=np.clip(a,0,1)
    im=Image.fromarray((a*255).astype("uint8")); b=io.BytesIO(); im.save(b,"JPEG",quality=r.randint(52,95)); b.seek(0); return Image.open(b).convert("L")
def ten(im): return torch.from_numpy(np.asarray(im,dtype=np.float32)/255.).unsqueeze(0)
class Pairs(Dataset):
    def __init__(self,N,V): self.N=list(N); self.V=V
    def __len__(self): return len(self.N)*self.V
    def __getitem__(self,i):
        n=self.N[i//self.V]; r=rng(n,i%self.V); c=crop(Image.open(path(n)).convert("L"),r); return ten(degrade(c,r)),ten(c),n
tl=DataLoader(Pairs(TR,8),BATCH,shuffle=True,num_workers=2,pin_memory=True); dl=DataLoader(Pairs(DV,2),BATCH,num_workers=2,pin_memory=True)
class C(nn.Module):
    def __init__(self,a,b): super().__init__(); self.n=nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.GroupNorm(8,b),nn.SiLU(),nn.Conv2d(b,b,3,padding=1),nn.GroupNorm(8,b),nn.SiLU())
    def forward(self,x): return self.n(x)
class UNet(nn.Module):
    def __init__(self,b=32):
        super().__init__(); self.e1=C(1,b);self.e2=C(b,2*b);self.e3=C(2*b,4*b);self.mid=C(4*b,8*b);self.d3=C(12*b,4*b);self.d2=C(6*b,2*b);self.d1=C(3*b,b);self.o=nn.Conv2d(b,1,1)
    def forward(self,x):
        a=self.e1(x);b=self.e2(F.max_pool2d(a,2));c=self.e3(F.max_pool2d(b,2));d=self.mid(F.max_pool2d(c,2))
        d=self.d3(torch.cat([F.interpolate(d,size=c.shape[-2:],mode="bilinear",align_corners=False),c],1));d=self.d2(torch.cat([F.interpolate(d,size=b.shape[-2:],mode="bilinear",align_corners=False),b],1));d=self.d1(torch.cat([F.interpolate(d,size=a.shape[-2:],mode="bilinear",align_corners=False),a],1))
        return torch.clamp(x+torch.tanh(self.o(d))*.5,0,1)
m=UNet().to(D); opt=torch.optim.AdamW(m.parameters(),LR,weight_decay=1e-5); scaler=torch.amp.GradScaler("cuda")
sx=torch.tensor([[-1.,0,1],[-2,0,2],[-1,0,1]],device=D).view(1,1,3,3); sy=sx.transpose(2,3)
def edge(x): return torch.sqrt(F.conv2d(x,sx,padding=1)**2+F.conv2d(x,sy,padding=1)**2+1e-6)
def loss(p,y): 
    px=F.l1_loss(p,y); ed=F.l1_loss(edge(p),edge(y)); return px+.3*ed,px,ed

In [ ]:
CFG={"datasetId":"deepscoresv2.dense.v2","archiveMd5":MD5,"model":"residual_unet_base32","patch":PATCH,"batch":BATCH,"epochs":EPOCHS,"lr":LR,"trainVariants":8,"devVariants":2,"loss":"l1+0.30*sobel_l1","seed":SEED,"heldOutTuning":False,"pretrainedWeights":False}
DIG=hashlib.sha256(json.dumps(CFG,sort_keys=True).encode()).hexdigest(); (O/"training_config.json").write_text(json.dumps({**CFG,"sha256":DIG},indent=2))
last,best=O/"last.pt",O/"best.pt"; start=0; bd=float("inf"); hist=[]
if last.exists():
    c=torch.load(last,map_location=D)
    if c["md5"]!=MD5 or c["cfg"]!=DIG: raise RuntimeError("Checkpoint identity mismatch")
    m.load_state_dict(c["model"]);opt.load_state_dict(c["opt"]);start=c["epoch"]+1;bd=c["best"];hist=c["hist"]
def epoch(loader,train):
    m.train(train); S=np.zeros(5)
    for x,y,_ in loader:
        x,y=x.to(D,non_blocking=True),y.to(D,non_blocking=True)
        if train: opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda",dtype=torch.float16): p=m(x); L,px,ed=loss(p,y)
        if train: scaler.scale(L).backward();scaler.unscale_(opt);torch.nn.utils.clip_grad_norm_(m.parameters(),1);scaler.step(opt);scaler.update()
        n=x.size(0);S+=np.array([L.item(),px.item(),ed.item(),F.mse_loss(p.detach(),y).item(),1])*n
    q=S[:4]/S[4]; return {"loss":float(q[0]),"pixel":float(q[1]),"edge":float(q[2]),"psnr":float(10*math.log10(1/max(q[3],1e-12)))}
x,y,_=next(iter(tl))
with torch.no_grad(),torch.autocast("cuda",dtype=torch.float16): smoke=F.l1_loss(m(x[:1].to(D)),y[:1].to(D)).item()
print("GPU:",torch.cuda.get_device_name(0),"smoke L1:",smoke)
for e in range(start,EPOCHS):
    a=epoch(tl,True)
    with torch.no_grad(): b=epoch(dl,False)
    hist.append({"epoch":e,"train":a,"development":b}); pay={"epoch":e,"model":m.state_dict(),"opt":opt.state_dict(),"best":min(bd,b["loss"]),"hist":hist,"md5":MD5,"cfg":DIG};torch.save(pay,last)
    if b["loss"]<bd:bd=b["loss"];torch.save(pay,best)
    (O/"run_history.json").write_text(json.dumps(hist,indent=2));print(hist[-1])
ev={"artifactType":"stage11_first_gpu_run_evidence","datasetId":"deepscoresv2.dense.v2","archiveMd5":MD5,"archiveChecksumVerified":True,"gpu":torch.cuda.get_device_name(0),"epochsCompleted":len(hist),"bestDevelopmentLoss":bd,"bestCheckpointExists":best.exists(),"lastCheckpointExists":last.exists(),"configSha256":DIG,"heldOutUsedForTuning":False,"productionInferenceAuthorized":False,"modelPublicationAuthorized":False,"stage12EntryAuthorized":False,"completedAtUnix":int(time.time())}
(O/"first_gpu_run_evidence.json").write_text(json.dumps(ev,indent=2));print(ev)

RUN_HELD_OUT_FINAL=False
if RUN_HELD_OUT_FINAL:
    fr=O/"FROZEN_CONFIG.json"
    if not fr.exists() or json.loads(fr.read_text()).get("configSha256")!=DIG: raise RuntimeError("Held-out requires matching frozen config")
    raise RuntimeError("Run final held-out evaluation only in the dedicated frozen-evaluation step after model selection.")
else: print("Held-out remains disabled during development.")